# Cleaning the Raw Data: California, 2016–2020

Produces `data/clean_ca_2016_2020.parquet`, which every other notebook loads with `src.data.load()`.

In [15]:
import sys; sys.path.append("..")
import glob
import re
import numpy as np
import pandas as pd

## 1. Load the 5 yearly files, keeping only California fires

In [16]:
files = sorted(glob.glob("../data/*_FPA_FOD_cons.csv"))
print(files)

# Filter each file to California as it loads
df = pd.concat((pd.read_csv(f, low_memory=False).query("STATE == 'CA'") for f in files),
               ignore_index=True)
print(df.shape)
df["FIRE_YEAR"].value_counts().sort_index()

['../data/2016_FPA_FOD_cons.csv', '../data/2017_FPA_FOD_cons.csv', '../data/2018_FPA_FOD_cons.csv', '../data/2019_FPA_FOD_cons.csv', '../data/2020_FPA_FOD_cons.csv']
(43561, 308)


FIRE_YEAR
2016     7882
2017     9537
2018     9488
2019     6454
2020    10200
Name: count, dtype: int64

## 2. First look

In [17]:
pd.set_option("display.max_rows", 400)
print(df["FIRE_SIZE_CLASS"].value_counts().sort_index())   # target balance
df.isna().mean().sort_values(ascending=False).head(30)     # most-missing columns

FIRE_SIZE_CLASS
A    27279
B    13678
C     1693
D      394
E      197
F      177
G      143
Name: count, dtype: int64


geometry                         1.000000
IALMIL_87                        1.000000
GACC_Fire Use Teams              1.000000
IAHSEF                           1.000000
IAULHS_89                        1.000000
IAPLHS_88                        1.000000
road_interstate_dis              0.999242
road_US_dis                      0.997681
ICS_209_PLUS_COMPLEX_JOIN_ID     0.994628
COMPLEX_NAME                     0.994582
MTBS_ID                          0.993274
MTBS_FIRE_NAME                   0.993274
road_other_dis                   0.993182
road_state_dis                   0.986410
ICS_209_PLUS_INCIDENT_JOIN_ID    0.983747
road_county_dis                  0.983586
NWCG_CAUSE_AGE_CATEGORY          0.973692
Evacuation                       0.915521
LOCAL_FIRE_REPORT_ID             0.844150
No_FireStation_1.0km             0.813801
FIRE_CODE                        0.803861
CONT_TIME                        0.576272
CONT_DATE                        0.550814
CONT_DOY                         0

## 3. Drop leakage and ID columns

- **Leakage:** anything only known after the fire (containment, cause, MTBS/ICS-209 links that only exist for large fires).
- **`_5D_` columns:** the 5-day windows are centered on ignition (2 days before to 2 days after), so they include post-ignition weather (Pourmohamad et al., 2024).
- `FIRE_SIZE` is kept for analysis only, never as a model input.

In [18]:
drop = [
    # Known only after the fire
    "CONT_DATE", "CONT_DOY", "CONT_TIME",
    "NWCG_CAUSE_CLASSIFICATION", "NWCG_GENERAL_CAUSE", "NWCG_CAUSE_AGE_CATEGORY",
    "MTBS_ID", "MTBS_FIRE_NAME", "ICS_209_PLUS_INCIDENT_JOIN_ID", "ICS_209_PLUS_COMPLEX_JOIN_ID",
    "COMPLEX_NAME",
    # IDs and names (no predictive meaning)
    "FOD_ID", "FPA_ID", "SOURCE_SYSTEM_TYPE", "SOURCE_SYSTEM", "NWCG_REPORTING_AGENCY",
    "NWCG_REPORTING_UNIT_ID", "NWCG_REPORTING_UNIT_NAME", "SOURCE_REPORTING_UNIT",
    "SOURCE_REPORTING_UNIT_NAME", "LOCAL_FIRE_REPORT_ID", "LOCAL_INCIDENT_ID",
    "FIRE_CODE", "FIRE_NAME",
]
drop += [c for c in df.columns if "_5D_" in c]   # post-ignition weather windows

df = df.drop(columns=drop, errors="ignore").copy()
print(df.shape)
print("Remaining _5D_ columns:", [c for c in df.columns if "_5D_" in c])   # should be []

(43561, 253)
Remaining _5D_ columns: []


## 4. Placeholder values and dates

In [19]:
# Treat -999 / -9999 style placeholders as missing
num_cols = df.select_dtypes("number").columns
df[num_cols] = df[num_cols].mask(df[num_cols] <= -999)

df["DISCOVERY_DATE"] = pd.to_datetime(df["DISCOVERY_DATE"], errors="coerce")
df["DISCOVERY_MONTH"] = df["DISCOVERY_DATE"].dt.month

/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/2192914080.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DISCOVERY_MONTH"] = df["DISCOVERY_DATE"].dt.month


## 5. Missing values: drop only columns that are mostly empty (gaps get filled later, on train only)

In [20]:
missing = df.isna().mean()
print(missing[missing > 0.5].sort_values(ascending=False))

df = df.loc[:, missing <= 0.5]
df = df.dropna(subset=["FIRE_SIZE_CLASS", "LATITUDE", "LONGITUDE"])
df.shape

GACC_Fire Use Teams     1.000000
IALMIL_87               1.000000
IAPLHS_88               1.000000
IAULHS_89               1.000000
IAHSEF                  1.000000
geometry                1.000000
road_interstate_dis     0.999242
road_US_dis             0.997681
road_other_dis          0.993182
road_state_dis          0.986410
road_county_dis         0.983586
Evacuation              0.915521
No_FireStation_1.0km    0.813801
dtype: float64


(43561, 241)

## 6. Target: merge the rarest size classes

In [21]:
merge = {"A": "A", "B": "B", "C": "C", "D": "D-E", "E": "D-E", "F": "F-G", "G": "F-G"}
df["TARGET"] = df["FIRE_SIZE_CLASS"].map(merge)
df["TARGET"].value_counts().sort_index()

/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/2609247854.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["TARGET"] = df["FIRE_SIZE_CLASS"].map(merge)


TARGET
A      27279
B      13678
C       1693
D-E      591
F-G      320
Name: count, dtype: int64

## 7. Percentile columns: text ranges → numbers

Labels like `"<10th"`, `"10th-30th"`, `">90th"` become the midpoint of the range (5, 20, … 95), so they keep their order.

In [22]:
pct_cols = [c for c in df.columns if c.endswith("_Percentile")]
print("Before:", df[pct_cols[0]].value_counts(dropna=False).to_dict())

def pct_to_number(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)
    s = str(v).strip()
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums:
        return np.nan
    if s.startswith("<"):
        return nums[0] / 2              # "<10th"  -> 5
    if s.startswith(">"):
        return (nums[0] + 100) / 2      # ">90th"  -> 95
    return sum(nums) / len(nums)        # "10th-30th" -> 20

for c in pct_cols:
    df[c] = df[c].map(pct_to_number)

print("After:", df[pct_cols[0]].value_counts(dropna=False).sort_index().to_dict())

Before: {'>90%': 43561}
After: {95.0: 43561}


## 8. Category codes → one-hot columns

`Land_Cover` and `EVT` are codes, not amounts. Keep the 20 most common codes each, lump the rest into "other".

In [23]:
cat_cols = [c for c in ["Land_Cover", "EVT"] if c in df.columns]
for c in cat_cols:
    top = df[c].value_counts().index[:20]
    df[c] = df[c].where(df[c].isin(top), "other").fillna("missing").astype(str)

df = pd.get_dummies(df, columns=cat_cols, prefix=["LC" if c == "Land_Cover" else "EVT" for c in cat_cols], dtype=float)
print([c for c in df.columns if c.startswith(("LC_", "EVT_"))])

['EVT_1km', 'LC_0.0', 'LC_11.0', 'LC_21.0', 'LC_22.0', 'LC_23.0', 'LC_24.0', 'LC_31.0', 'LC_41.0', 'LC_42.0', 'LC_43.0', 'LC_52.0', 'LC_71.0', 'LC_81.0', 'LC_82.0', 'LC_90.0', 'LC_95.0', 'EVT_7027.0', 'EVT_7028.0', 'EVT_7030.0', 'EVT_7032.0', 'EVT_7043.0', 'EVT_7092.0', 'EVT_7098.0', 'EVT_7105.0', 'EVT_7110.0', 'EVT_7296.0', 'EVT_7297.0', 'EVT_7298.0', 'EVT_7299.0', 'EVT_7901.0', 'EVT_7904.0', 'EVT_7913.0', 'EVT_7914.0', 'EVT_7984.0', 'EVT_9301.0', 'EVT_9337.0', 'EVT_other']


## 9. Aspect is a compass direction → sine and cosine

**Aspect → sin/cos:** Aspect is a compass direction (0°–360°), and 359° and 1° both face north even though the numbers are far apart, so we convert it to `cos` (north = +1, south = −1) and `sin` (east = +1, west = −1). That way the model sees nearby directions as similar.

In [24]:
for c, s in [("Aspect", "Slope"), ("Aspect_1km", "Slope_1km")]:
    if c in df.columns:
        rad = np.deg2rad(df[c])
        df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)
        if s in df.columns:
            flat = df[s] < 2                      # nearly flat: no real facing direction
            df.loc[flat, [c + "_sin", c + "_cos"]] = 0

/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/4032990377.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)
/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/4032990377.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)
/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/4032990377.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result 

**Flat ground fix:** Flat ground doesn't face any direction, but the data records it as 0°, which would be read as "north-facing." During EDA this made north-facing slopes look much safer than they really are, because flat areas like farms and towns rarely have large fires. So for nearly flat ground (slope < 2°), we set both `sin` and `cos` to 0, meaning "no direction."

## 10. Regions for the location split

California is small, so use **0.5° × 0.5° grid blocks** (about 50 km) instead of 1°. Every block goes entirely into train or test.

In [25]:
df["REGION"] = (np.floor(df["LATITUDE"] / 0.5).astype(int).astype(str) + "_" +
                np.floor(df["LONGITUDE"] / 0.5).astype(int).astype(str))

print(df["REGION"].nunique(), "regions")
# Are large fires spread across many regions, or concentrated in a few?
fg = df.groupby("REGION")["TARGET"].apply(lambda s: (s == "F-G").sum()).sort_values(ascending=False)
print((fg > 0).sum(), "regions contain at least one F-G fire")
fg.head(10)

197 regions
109 regions contain at least one F-G fire


/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/2191540303.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["REGION"] = (np.floor(df["LATITUDE"] / 0.5).astype(int).astype(str) + "_" +


REGION
83_-247    9
67_-236    8
67_-235    7
75_-240    7
68_-238    7
76_-245    7
79_-246    7
72_-241    7
72_-238    6
77_-245    6
Name: TARGET, dtype: int64

## 11. Check the feature groups and the folds

In [26]:
from src.data import feature_sets, cv_folds

sets = feature_sets(df)
for name, cols in sets.items():
    print(name, len(cols), "features")

# Class mix in each test fold (should be roughly similar; watch the F-G counts)
pd.DataFrame({f"fold {i}": te["TARGET"].value_counts()
              for i, (tr, te) in enumerate(cv_folds(df))}).sort_index()

Skipping non-numeric columns: ['NDVI_min', 'NDVI_max', 'NDVI_mean', 'MOD_NDVI_12m', 'EVT_1km']
Skipping non-numeric columns: ['NDVI_min', 'NDVI_max', 'NDVI_mean', 'MOD_NDVI_12m', 'EVT_1km']
model1_weather 36 features
model2_weather_veg 74 features
model3_all 86 features


,fold 0,fold 1,fold 2,fold 3,fold 4
TARGET,,,,,
A,5581,5467,5499,5395,5337
B,2667,2692,2688,2795,2836
C,290,366,346,341,350
D-E,125,123,115,116,112
F-G,50,64,64,65,77


## 12. Keep only what's needed and save

In [27]:
extra = ["TARGET", "REGION", "FIRE_SIZE", "FIRE_SIZE_CLASS", "FIRE_YEAR", "STATE",
         "DISCOVERY_DATE", "Population", "GAP_Prity", "EPL_MOBILE"]
keep = list(dict.fromkeys(sets["model3_all"] + [c for c in extra if c in df.columns]))
df = df[keep].copy()
print(df.shape)

df.to_parquet("../data/clean_ca_2016_2020.parquet")

(43561, 96)


## 13. Sanity check: reload the saved file

In [28]:
from src.data import load
check = load()
print(check.shape)
check["TARGET"].value_counts().sort_index()

(43561, 96)


TARGET
A      27279
B      13678
C       1693
D-E      591
F-G      320
Name: count, dtype: int64